# Week 1 Assignment: Worked Answers

This instructor copy solves the Customer Purchase Insights assignment using only Python's standard library. The solution uses CSV file handling, type conversion, lists, dictionaries, sets, tuples, loops, conditions, functions, a default argument, and a lambda sort key.

Run cells from top to bottom. Start with the small sample tests; then run the same functions on the supplied CSV.

## 1. Imports and file path

`csv.DictReader` uses the CSV header to label each field. `Path.cwd()` shows the current working directory; the fallback below handles running the notebook either inside the Week 1 folder or from the workspace root.

In [ ]:
import csv
from pathlib import Path

DATA_PATH = Path("customer_details.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("Week - 1 Python Basics") / "customer_details.csv"

print("Working directory:", Path.cwd())
print("CSV path:", DATA_PATH.resolve())
print("CSV found:", DATA_PATH.exists())

## 2. Inspect the header and a few rows

Opening the file with a `with` statement closes it when the block ends. `newline=""` is the recommended setting when opening a CSV file with Python's `csv` module.

In [ ]:
with DATA_PATH.open("r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    print("Headers:", reader.fieldnames)
    for row_number, row in enumerate(reader):
        if row_number == 3:
            break
        print(row)

## 3. Load records

Each row is a dictionary from column name to cell text. Numeric-looking fields remain strings, so calculations must convert them explicitly. The function returns the data instead of printing it, allowing other functions to reuse it.

In [ ]:
def load_customers(path):
    """Read a customer CSV and return its rows as dictionaries."""
    with Path(path).open("r", encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


customers = load_customers(DATA_PATH)
print("Number of records:", len(customers))
print("First record fields:", list(customers[0].keys()) if customers else [])

## 4. Count categories and collect unique values

`counts.get(category, 0)` reads the current count or supplies zero for a category seen for the first time. The set comprehension keeps one copy of each category. Sets are for uniqueness and membership, so sort values when a stable display order is needed.

In [ ]:
def category_counts(customers):
    """Return a dictionary mapping each category to a record count."""
    counts = {}
    for customer in customers:
        category = customer["Category"]
        counts[category] = counts.get(category, 0) + 1
    return counts


unique_categories = {customer["Category"] for customer in customers}
print("Unique categories:", sorted(unique_categories))
print("Counts by category:", category_counts(customers))

## 5. Average purchase amount by category

Keep totals and counts in separate dictionaries. After the loop, divide each total by the corresponding count. When the input is empty, both dictionaries remain empty and the returned result is `{}`; no division occurs.

In [ ]:
def average_purchase_by_category(customers):
    """Return the mean purchase amount for each category."""
    totals = {}
    counts = {}

    for customer in customers:
        category = customer["Category"]
        amount = float(customer["Purchase Amount (USD)"])
        totals[category] = totals.get(category, 0.0) + amount
        counts[category] = counts.get(category, 0) + 1

    return {
        category: totals[category] / counts[category]
        for category in totals
    }


average_by_category = average_purchase_by_category(customers)
print(average_by_category)

## 6. Filter high-value purchases

The default threshold is used when the caller omits the second argument. The list comprehension evaluates the condition for every row and retains only matching records. Amounts are converted before comparison because CSV values are text.

In [ ]:
def high_value_customers(customers, minimum_amount=80):
    """Return rows at or above the purchase threshold."""
    return [
        customer
        for customer in customers
        if float(customer["Purchase Amount (USD)"]) >= minimum_amount
    ]


high_value = high_value_customers(customers)
print("Purchases of at least $80:", len(high_value))

## 7. Sort top purchases

Each purchase summary is a tuple with a fixed shape: amount, item, and location. `sorted()` returns a new list. The lambda tells it to sort by the first tuple value, highest first; `limit` controls how many results are returned.

In [ ]:
def top_purchases(customers, limit=5):
    """Return the top purchases as (amount, item, location) tuples."""
    purchases = []
    for customer in customers:
        purchase = (
            float(customer["Purchase Amount (USD)"]),
            customer["Item Purchased"],
            customer["Location"],
        )
        purchases.append(purchase)

    ranked_purchases = sorted(
        purchases,
        key=lambda purchase: purchase[0],
        reverse=True,
    )
    return ranked_purchases[:limit]


for purchase in top_purchases(customers):
    print(purchase)

## 8. Assemble and display the report

`build_report` coordinates the calculations and returns one dictionary. The display loop sorts categories for readability and formats average amounts to two decimal places. Separating computation from presentation makes the functions easier to test.

In [ ]:
def build_report(path):
    """Load customer rows and assemble the requested report values."""
    rows = load_customers(path)
    return {
        "customer_count": len(rows),
        "unique_categories": {row["Category"] for row in rows},
        "category_counts": category_counts(rows),
        "average_by_category": average_purchase_by_category(rows),
        "high_value_count": len(high_value_customers(rows)),
        "top_purchases": top_purchases(rows),
    }


report = build_report(DATA_PATH)
print(f"Customer records: {report['customer_count']}")
print("\nRecords and average purchase by category:")
for category in sorted(report["category_counts"]):
    count = report["category_counts"][category]
    average = report["average_by_category"][category]
    print(f"- {category}: {count} records, average ${average:.2f}")
print(f"\nPurchases of at least $80: {report['high_value_count']}")
print("Top purchases (amount, item, location):")
for purchase in report["top_purchases"]:
    print(f"- ${purchase[0]:.2f}, {purchase[1]}, {purchase[2]}")

## 9. Verify with a small sample

Small, known inputs let us check behavior before relying on the full dataset. Assertions stop with an error if an expected result is wrong; if the cell finishes without an error, all checks passed.

In [ ]:
sample_customers = [
    {"Category": "Books", "Purchase Amount (USD)": "10", "Item Purchased": "Notebook", "Location": "Pune", "Subscription Status": "Yes"},
    {"Category": "Food", "Purchase Amount (USD)": "5", "Item Purchased": "Tea", "Location": "Delhi", "Subscription Status": "No"},
    {"Category": "Books", "Purchase Amount (USD)": "20", "Item Purchased": "Novel", "Location": "Pune", "Subscription Status": "Yes"},
]

assert category_counts(sample_customers) == {"Books": 2, "Food": 1}
assert average_purchase_by_category(sample_customers) == {"Books": 15.0, "Food": 5.0}
assert len(high_value_customers(sample_customers)) == 0
assert len(high_value_customers(sample_customers, minimum_amount=10)) == 2
assert top_purchases(sample_customers, limit=2) == [
    (20.0, "Novel", "Pune"),
    (10.0, "Notebook", "Pune"),
]
assert average_purchase_by_category([]) == {}
assert category_counts([]) == {}
print("All sample-data tests passed.")

## Optional extension examples

These examples connect to flexible function arguments from the functions session. They are not needed for the core assignment.

In [ ]:
def average_of_values(*values):
    """Return the mean of any number of numeric positional arguments."""
    if not values:
        raise ValueError("provide at least one value")
    return sum(values) / len(values)


def print_summary(**details):
    """Print named report details supplied as keyword arguments."""
    for label, value in details.items():
        print(f"{label}: {value}")


print(average_of_values(10, 20, 30))
print_summary(records=3, categories=2)